In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 트랜스파일 기본 설정 및 구성 옵션


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
추상 Circuit은 QPU가 제한된 기저 게이트(basis gate) 집합만 지원하며 임의의 연산을 실행할 수 없기 때문에 트랜스파일이 필요합니다. Transpiler의 역할은 임의의 Circuit을 지정된 QPU에서 실행할 수 있도록 변환하는 것입니다. 이는 Circuit을 지원되는 기저 Gate로 변환하고, Circuit의 연결성이 QPU의 연결성과 일치하도록 필요에 따라 SWAP Gate를 도입함으로써 이루어집니다.

[패스 매니저로 트랜스파일하기](transpile-with-pass-managers)에서 설명한 바와 같이, [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 함수를 사용하여 [패스 매니저](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager)를 생성하고, Circuit 또는 Circuit 목록을 해당 [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) 메서드에 전달하여 트랜스파일할 수 있습니다. `generate_preset_pass_manager`를 호출할 때 최적화 수준과 Backend만 전달하여 나머지 옵션에 대한 기본값을 사용하거나, 트랜스파일을 세밀하게 조정하기 위해 함수에 추가 인수를 전달할 수도 있습니다.

## 매개변수 없이 기본 사용
이 예시에서는 추가 매개변수를 지정하지 않고 Circuit과 대상 QPU를 Transpiler에 전달합니다.

Circuit을 생성하고 결과를 확인합니다:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Circuit을 트랜스파일하고 결과를 확인합니다:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## 사용 가능한 모든 매개변수
다음은 [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 함수에서 사용할 수 있는 모든 매개변수입니다. 매개변수는 두 가지 클래스로 나뉩니다: 컴파일 대상을 설명하는 것과 Transpiler의 동작 방식에 영향을 미치는 것입니다.

`optimization_level`을 제외한 모든 매개변수는 선택 사항입니다. 전체 세부 정보는 [Transpiler API 문서](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api)를 참조하세요.

- `optimization_level` (int) - Circuit에 대해 수행할 최적화의 정도입니다. 0에서 3 사이의 정수입니다. 수준이 높을수록 더 최적화된 Circuit이 생성되지만, 트랜스파일 시간이 더 오래 걸립니다. 자세한 내용은 [Transpiler 최적화 수준 설정](set-optimization)을 참조하세요.

### 컴파일 대상을 설명하는 데 사용되는 매개변수
이 인수들은 Circuit 실행을 위한 대상 QPU를 설명하며, QPU의 커플링 맵(Qubit의 연결성을 설명하는 정보), QPU에서 지원하는 기저 Gate, Gate의 오류율 등의 정보를 포함합니다.

이러한 매개변수 중 다수는 [트랜스파일에 일반적으로 사용되는 매개변수](common-parameters)에서 자세히 설명되어 있습니다.

<details>
  <summary>
    **QPU (`Backend`) 매개변수**
  </summary>

**Backend 매개변수** - `backend`를 지정하면 `target`이나 다른 Backend 옵션을 지정할 필요가 없습니다. 마찬가지로, `target`을 지정하면 `backend`나 다른 Backend 옵션을 지정할 필요가 없습니다.
- `backend` (Backend) - 설정되면 Transpiler가 입력 Circuit을 해당 장치에 맞게 컴파일합니다. `coupling_map`과 같이 이 설정에 영향을 미치는 다른 옵션이 설정된 경우, 해당 옵션이 `backend`의 설정을 재정의합니다.
- `target` (Target) - Backend Transpiler 대상입니다. 일반적으로 backend 인수의 일부로 지정되지만, Target 객체를 수동으로 생성한 경우 여기에서 지정할 수 있습니다. 이는 `backend`의 대상을 재정의합니다.
- `backend_properties` (BackendProperties) - QPU에서 반환되는 속성으로, Gate 오류, 읽기 오류, Qubit 결맞음 시간 등에 대한 정보를 포함합니다. `backend.properties()`를 실행하여 이 정보를 제공하는 QPU를 찾을 수 있습니다.
- `timing_constraints` (Dict[str, int] | None) - 명령 시간 해상도에 대한 선택적 제어 하드웨어 제한입니다. 이 정보는 QPU 구성에서 제공됩니다. QPU에 명령 시간 할당에 대한 제한이 없으면 `timing_constraints`는 `None`이며 조정이 수행되지 않습니다. QPU는 다음과 같은 제한 집합을 보고할 수 있습니다:
    - `granularity`: dt 단위로 최소 펄스 Gate 해상도를 나타내는 정수 값입니다. 사용자 정의 펄스 Gate의 지속 시간은 이 세분성 값의 배수여야 합니다.
    - `min_length`: dt 단위로 최소 펄스 Gate 길이를 나타내는 정수 값입니다. 사용자 정의 펄스 Gate는 이 길이보다 길어야 합니다.
    - `pulse_alignment`: Gate 명령 시작 시간의 시간 해상도를 나타내는 정수 값입니다. Gate 명령은 이 값의 배수인 시간에 시작해야 합니다.
    - `acquire_alignment`: 측정 명령 시작 시간의 시간 해상도를 나타내는 정수 값입니다. 측정 명령은 이 값의 배수인 시간에 시작해야 합니다.
</details>

<details>
  <summary>
    **레이아웃 및 토폴로지 매개변수**
  </summary>

- `basis_gates` (List[str] | None) - 언롤(unroll)할 기저 Gate 이름 목록입니다. 예를 들어 ['u1', 'u2', 'u3', 'cx']. `None`이면 언롤하지 않습니다.
- `coupling_map` (CouplingMap | List[List[int]]) - 매핑에서 대상으로 사용할 방향성 커플링 맵(사용자 정의 가능)입니다. 커플링 맵이 대칭인 경우 양방향을 모두 지정해야 합니다. 다음 형식이 지원됩니다:
    - CouplingMap 인스턴스
    - List - 인접 행렬로 제공해야 하며, 각 항목은 QPU에서 지원하는 모든 방향성 2-Qubit 상호작용을 지정합니다. 예: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Circuit 연산에서 펄스 스케줄로의 매핑입니다. `None`이면 QPU의 `instruction_schedule_map`이 사용됩니다.
</details>

### Transpiler 동작 방식에 영향을 미치는 매개변수
이 매개변수들은 특정 트랜스파일 단계에 영향을 미칩니다. 일부는 여러 단계에 영향을 미칠 수 있지만, 간결성을 위해 하나의 단계에만 나열했습니다. 사용하려는 Qubit에 대한 `initial_layout`과 같이 인수를 지정하면 해당 값이 변경할 수 있는 모든 패스를 재정의합니다. 즉, Transpiler는 수동으로 지정한 것을 변경하지 않습니다. 특정 단계에 대한 자세한 내용은 [Transpiler 단계](transpiler-stages)를 참조하세요.

<details>
  <summary>
    **초기화 단계**
  </summary>

- `hls_config` (HLSConfig) - `HighLevelSynthesis` 변환 패스에 직접 전달되는 선택적 구성 클래스 `HLSConfig`입니다. 이 구성 클래스를 통해 다양한 고수준 객체에 대한 합성 알고리즘 목록 및 해당 매개변수를 지정할 수 있습니다.
- `init_method` (str) - 초기화 단계에 사용할 플러그인 이름입니다. 기본적으로 외부 플러그인은 사용되지 않습니다. 스테이지 이름 인수에 `init`를 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다.
- `unitary_synthesis_method` (str) - 사용할 유니터리 합성 방법의 이름입니다. 기본적으로 `default`가 사용됩니다. `unitary_synthesis_plugin_names()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다.
- `unitary_synthesis_plugin_config` (dict) - 유니터리 합성 플러그인에 직접 전달되는 선택적 구성 딕셔너리입니다. 기본적으로 이 설정은 기본 유니터리 합성 방법이 사용자 정의 구성을 사용하지 않기 때문에 효과가 없습니다. 사용자 정의 구성 적용은 `unitary_synthesis` 인수로 유니터리 합성 플러그인이 지정된 경우에만 필요합니다. 이는 각 유니터리 합성 플러그인마다 다르므로, 이 옵션 사용 방법은 해당 플러그인의 문서를 참조하세요.
</details>

<details>
  <summary>
    **레이아웃 단계**
  </summary>

- `initial_layout` (Layout | Dict | List) - 물리적 Qubit에 대한 가상 Qubit의 초기 위치입니다. 이 레이아웃으로 Circuit이 `coupling_map` 제약 조건과 호환되면 사용됩니다. Transpiler가 스왑이나 다른 방법으로 Qubit을 재배열할 수 있으므로 최종 레이아웃이 동일하다는 보장은 없습니다. 전체 세부 정보는 [초기 레이아웃 섹션](common-parameters#initial-layout)을 참조하세요.
- `layout_method` (str) - 레이아웃 선택 패스의 이름입니다 (`default`, `dense`, `sabre`, `trivial`). 레이아웃 단계에 사용할 외부 플러그인 이름일 수도 있습니다. `stage_name` 인수에 `layout`을 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다. 기본값은 `sabre`입니다.
</details>

<details>
  <summary>
    **라우팅 단계**
  </summary>

- `routing_method` (str) - 라우팅 패스의 이름입니다 (`basic`, `lookahead`, `default`, `sabre`, `none`). 라우팅 단계에 사용할 외부 플러그인 이름일 수도 있습니다. `stage_name` 인수에 `routing`을 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다. 기본값은 `sabre`입니다.
</details>

<details>
  <summary>
    **변환 단계**
  </summary>

- `translation_method` (str) - 변환 패스의 이름입니다 (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). 변환 단계에 사용할 외부 플러그인 이름일 수도 있습니다. `stage_name` 인수에 `translation`을 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다. 기본값은 `translator`입니다.
</details>

<details>
  <summary>
    **최적화 단계**
  </summary>

- `approximation_degree` (float, 0-1 범위 | None) - Circuit 근사에 사용되는 휴리스틱 다이얼입니다 (1.0 = 근사 없음, 0.0 = 최대 근사). 기본값은 1.0입니다. `None`을 지정하면 근사 수준이 보고된 오류율로 설정됩니다. 자세한 내용은 [근사 수준 섹션](common-parameters#approx-degree)을 참조하세요.
- `optimization_method` (str) - 최적화 단계에 사용할 플러그인 이름입니다. 기본적으로 외부 플러그인은 사용되지 않습니다. `stage_name` 인수에 `optimization`을 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다.
</details>

<details>
  <summary>
    **스케줄링 단계**
  </summary>

- `scheduling_method` (str) - 스케줄링 패스의 이름입니다. 스케줄링 단계에 사용할 외부 플러그인 이름일 수도 있습니다. `stage_name` 인수에 `scheduling`을 사용하여 `list_stage_plugins()`를 실행하면 설치된 플러그인 목록을 볼 수 있습니다.
  * 'as_soon_as_possible': Qubit 리소스에서 가능한 한 빨리 명령을 탐욕적으로 스케줄링합니다 (별칭: `asap`).
  * 'as_late_as_possible': 명령을 늦게 스케줄링합니다. 즉, 가능한 한 Qubit을 기저 상태로 유지합니다 (별칭: `alap`). 이것이 기본값입니다.
</details>

<details>
  <summary>
    **Transpiler 실행**
  </summary>

- `seed_transpiler` (int) - Transpiler의 확률적 부분에 대한 난수 시드를 설정합니다.
</details>

위의 매개변수를 지정하지 않으면 다음 기본값이 사용됩니다. 자세한 내용은 메서드의 [API 참조 페이지](../api/qiskit/transpiler_preset)를 참조하세요:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>